# X-REACH — DQN Training (5-fold, per-fold evaluation, backend-ready export)

Trains DQN on the 5 folds produced by `X-REACH_Full_Preprocessing_Pipeline.ipynb`
(`rl_environment.pkl`), evaluates every fold's own metrics separately, then
lets you download the best fold's trained model in the exact format the
X-REACH backend expects.

Includes the regularization fix discussed earlier (dropout + L2 weight
decay) aimed at the non-monotonic single-feature behavior (e.g. `age`)
found in the currently deployed DQN artifact -- caused by a nonlinear
network overfitting weak-signal features on a small dataset.

**How to use:** Runtime -> Run all. When prompted, upload `rl_environment.pkl`
(from the preprocessing notebook). At the end it triggers a browser download
of `weights.json` and `metadata.json` for the best fold -- no ONNX, no
`onnxruntime-node`, no PyTorch needed anywhere near the backend.


In [ ]:
import os
import random
import pickle
import json
from datetime import datetime, timezone
from collections import deque

import numpy as np
import pandas as pd

import torch
import torch.nn as nn
import torch.optim as optim

from sklearn.metrics import (
    accuracy_score, precision_score, recall_score, f1_score,
    confusion_matrix, roc_auc_score,
)
from sklearn.metrics.pairwise import cosine_similarity

import warnings
warnings.filterwarnings("ignore")

SEED = 42
random.seed(SEED)
np.random.seed(SEED)
torch.manual_seed(SEED)

DEVICE = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print("Device:", DEVICE)


## Load the 5-fold RL environment

Each fold is `{fold, states_train, labels_train, states_test, labels_test,
state_size, action_size}`. `states_train` already includes the augmented
rows (if you ran the preprocessing notebook); `states_test` is always real,
unaugmented data.


In [ ]:
ENV_PATH = "rl_environment.pkl"

if not os.path.exists(ENV_PATH):
    try:
        from google.colab import files
        print("Please upload rl_environment.pkl")
        uploaded = files.upload()
        ENV_PATH = list(uploaded.keys())[0]
    except ImportError:
        raise FileNotFoundError(f"{ENV_PATH} not found. Place it next to this notebook.")

with open(ENV_PATH, "rb") as f:
    folds = pickle.load(f)

print(f"Loaded {len(folds)} folds")
for fold in folds:
    print(f"  Fold {fold['fold']}: states_train {fold['states_train'].shape}, "
          f"states_test {fold['states_test'].shape}, state_size {fold['state_size']}")

STATE_SIZE = folds[0]["state_size"]

# Canonical feature order (see backend/utils/featureContract.js) -- for metadata.json only.
CANONICAL_FEATURE_ORDER = [
    "thalach", "restecg", "oldpeak", "slope", "age",
    "sex", "cp", "exang", "trestbps", "fbs",
]


## Hyperparameters & network

Same architecture as the currently deployed model
(`15 -> 128 -> 256 -> 128 -> ACTION_SIZE`), with **dropout** added after each
hidden layer and **L2 weight decay** added to the optimizer -- both aimed
specifically at reducing overfitting on weak-signal features (per the HRFLM
global importance report, `age` ranks 8th of 10 features; a plain,
undropped, undecayed network is prone to fitting noise on features like
that with only ~500-1000 training rows per fold).


In [ ]:
NUM_EPISODES = 300
MAX_STEPS = 300
GAMMA = 0.99
LEARNING_RATE = 0.001
EPSILON_START = 1.0
EPSILON_MIN = 0.01
EPSILON_DECAY = 0.995
BATCH_SIZE = 64
MEMORY_SIZE = 10000
TARGET_UPDATE = 10
WEIGHT_DECAY = 1e-4    # NEW -- L2 regularization
DROPOUT = 0.2          # NEW

NUM_FEATURES = STATE_SIZE
ACTION_SIZE = NUM_FEATURES * 3
ACTION_INCREASE, ACTION_DECREASE, ACTION_HOLD = 1, -1, 0
DELTA = 0.05


class DiseasePredictionEnvironment:
    def __init__(self, states, labels):
        self.original_states = np.array(states, dtype=np.float32)
        self.labels = np.array(labels, dtype=np.int64)
        self.num_samples = len(self.original_states)
        self.reset()

    def reset(self):
        self.current_index = np.random.randint(0, self.num_samples)
        self.current_state = self.original_states[self.current_index].copy()
        self.steps = 0
        return self.current_state.copy()

    def modify_state(self, state, action):
        new_state = state.copy()
        feature, operation = action // 3, action % 3
        if operation == ACTION_INCREASE:
            new_state[feature] += DELTA
        elif operation == ACTION_DECREASE:
            new_state[feature] -= DELTA
        return new_state

    def nearest_patient(self, modified_state):
        similarities = cosine_similarity(modified_state.reshape(1, -1), self.original_states)[0]
        best_index = np.argmax(similarities)
        return best_index, similarities[best_index]

    def step(self, action):
        self.steps += 1
        modified_state = self.modify_state(self.current_state, action)
        next_index, similarity = self.nearest_patient(modified_state)
        next_state = self.original_states[next_index].copy()
        label = self.labels[next_index]

        if label == 1:
            reward, done = 10.0, True
        else:
            reward, done = -1.0, False
        if self.steps >= MAX_STEPS:
            done = True

        self.current_index, self.current_state = next_index, next_state.copy()
        return next_state, reward, done, {"label": int(label)}

    def sample_action(self):
        return random.randint(0, ACTION_SIZE - 1)


class ReplayMemory:
    def __init__(self, capacity):
        self.memory = deque(maxlen=capacity)

    def push(self, state, action, reward, next_state, done):
        self.memory.append((np.array(state, dtype=np.float32), int(action), float(reward),
                             np.array(next_state, dtype=np.float32), bool(done)))

    def sample(self, batch_size):
        batch = random.sample(self.memory, batch_size)
        states, actions, rewards, next_states, dones = zip(*batch)
        return (torch.FloatTensor(np.array(states)).to(DEVICE),
                torch.LongTensor(np.array(actions)).unsqueeze(1).to(DEVICE),
                torch.FloatTensor(np.array(rewards)).unsqueeze(1).to(DEVICE),
                torch.FloatTensor(np.array(next_states)).to(DEVICE),
                torch.FloatTensor(np.array(dones)).unsqueeze(1).to(DEVICE))

    def __len__(self):
        return len(self.memory)


class DeepQNetwork(nn.Module):
    """15 -> 128 -> 256 -> 128 -> ACTION_SIZE, matching the deployed model's
    shape, with dropout added after each ReLU (NEW)."""
    def __init__(self, input_dim, output_dim):
        super().__init__()
        self.model = nn.Sequential(
            nn.Linear(input_dim, 128), nn.ReLU(), nn.Dropout(DROPOUT),
            nn.Linear(128, 256), nn.ReLU(), nn.Dropout(DROPOUT),
            nn.Linear(256, 128), nn.ReLU(), nn.Dropout(DROPOUT),
            nn.Linear(128, output_dim),
        )

    def forward(self, x):
        return self.model(x)


class DQNAgent:
    def __init__(self):
        self.policy_network = DeepQNetwork(STATE_SIZE, ACTION_SIZE).to(DEVICE)
        self.target_network = DeepQNetwork(STATE_SIZE, ACTION_SIZE).to(DEVICE)
        self.target_network.load_state_dict(self.policy_network.state_dict())
        self.target_network.eval()
        self.optimizer = optim.Adam(self.policy_network.parameters(), lr=LEARNING_RATE,
                                     weight_decay=WEIGHT_DECAY)
        self.criterion = nn.SmoothL1Loss()
        self.memory = ReplayMemory(MEMORY_SIZE)
        self.epsilon = EPSILON_START

    def act(self, state):
        if random.random() < self.epsilon:
            return random.randint(0, ACTION_SIZE - 1)
        state_t = torch.FloatTensor(state).unsqueeze(0).to(DEVICE)
        with torch.no_grad():
            q_values = self.policy_network(state_t)
        return torch.argmax(q_values).item()

    def remember(self, *args):
        self.memory.push(*args)

    def replay(self):
        if len(self.memory) < BATCH_SIZE:
            return None
        states, actions, rewards, next_states, dones = self.memory.sample(BATCH_SIZE)
        current_q = self.policy_network(states).gather(1, actions)
        with torch.no_grad():
            next_q = self.target_network(next_states).max(1)[0].unsqueeze(1)
            target_q = rewards + (1 - dones) * GAMMA * next_q
        loss = self.criterion(current_q, target_q)
        self.optimizer.zero_grad()
        loss.backward()
        torch.nn.utils.clip_grad_norm_(self.policy_network.parameters(), 1.0)
        self.optimizer.step()
        return loss.item()

    def update_target(self):
        self.target_network.load_state_dict(self.policy_network.state_dict())

    def decay_epsilon(self):
        self.epsilon = max(self.epsilon * EPSILON_DECAY, EPSILON_MIN)


def train_one_episode(agent, env):
    state, done, episode_reward, steps = env.reset(), False, 0.0, 0
    while not done and steps < MAX_STEPS:
        action = agent.act(state)
        next_state, reward, done, _ = env.step(action)
        agent.remember(state, action, reward, next_state, done)
        agent.replay()
        state = next_state
        episode_reward += reward
        steps += 1
    agent.update_target()
    agent.decay_epsilon()
    return episode_reward


## Train all 5 folds

Each fold gets its own freshly-initialized agent, trained only on that
fold's `states_train` (which already includes the augmented rows), and
evaluated only on that fold's own `states_test` (always real data).


In [ ]:
def train_dqn_fold(fold_data):
    env = DiseasePredictionEnvironment(fold_data["states_train"], fold_data["labels_train"])
    agent = DQNAgent()
    for episode in range(NUM_EPISODES):
        reward = train_one_episode(agent, env)
        if (episode + 1) % TARGET_UPDATE == 0:
            agent.update_target()
        if (episode + 1) % 50 == 0:
            print(f"    episode {episode+1}/{NUM_EPISODES} | reward {reward:.1f} | eps {agent.epsilon:.3f}")
    return agent


print("=" * 70)
print(f"TRAINING DQN -- {len(folds)} folds")
print("=" * 70)

dqn_agents = []
for fold_data in folds:
    print(f"\nFold {fold_data['fold']}")
    agent = train_dqn_fold(fold_data)
    dqn_agents.append(agent)

print("\nDQN training complete for all folds.")


## Evaluate each fold separately

Prints and stores per-fold Accuracy / Precision / Recall / F1 / Specificity
/ ROC-AUC / confusion matrix, exactly like the original notebook's
evaluation cell, plus saves `dqn_evaluation_results.csv`.


In [ ]:
all_metrics = []

print("=" * 70)
print("EVALUATING EACH FOLD")
print("=" * 70)

for fold_data, agent in zip(folds, dqn_agents):
    fold_number = fold_data["fold"]
    agent.policy_network.eval()

    X_test = np.array(fold_data["states_test"], dtype=np.float32)
    y_test = np.array(fold_data["labels_test"], dtype=np.int64)

    predictions, probabilities = [], []
    with torch.no_grad():
        for state in X_test:
            state_tensor = torch.FloatTensor(state).unsqueeze(0).to(DEVICE)
            q_values = agent.policy_network(state_tensor)
            probs = torch.softmax(q_values[:, :2], dim=1)
            pred = torch.argmax(q_values[:, :2], dim=1).item()
            predictions.append(pred)
            probabilities.append(probs[0, 1].item())

    agent.policy_network.train()

    acc = accuracy_score(y_test, predictions)
    prec = precision_score(y_test, predictions, zero_division=0)
    rec = recall_score(y_test, predictions, zero_division=0)
    f1 = f1_score(y_test, predictions, zero_division=0)
    cm = confusion_matrix(y_test, predictions)

    if cm.shape == (2, 2):
        tn, fp, fn, tp = cm.ravel()
        specificity = tn / (tn + fp) if (tn + fp) > 0 else 0.0
    else:
        specificity = 0.0

    try:
        auc = roc_auc_score(y_test, probabilities)
    except ValueError:
        auc = float("nan")

    metrics = {"Fold": fold_number, "Accuracy": acc, "Precision": prec, "Recall": rec,
               "F1": f1, "Specificity": specificity, "ROC_AUC": auc}
    all_metrics.append(metrics)

    print(f"\nFold {fold_number}")
    print(metrics)
    print("Confusion Matrix:")
    print(cm)

results_df = pd.DataFrame(all_metrics)
print("\n" + "=" * 70)
print("AVERAGE PERFORMANCE ACROSS ALL FOLDS")
print("=" * 70)
print(results_df.mean(numeric_only=True))

results_df.to_csv("dqn_evaluation_results.csv", index=False)
print("\nSaved dqn_evaluation_results.csv")
results_df


## Export the best fold for the backend (no ONNX)

Instead of exporting to ONNX, this dumps the trained network's weights as
plain JSON -- a list of layers (`linear` / `relu`), each `linear` layer
giving its weight matrix and bias vector as plain nested arrays -- plus
`metadata.json`. Dropout layers are skipped entirely (dropout is only
active during training, not inference, so it contributes nothing to the
forward pass and doesn't need representing at all). This is the same
pattern already used for Q-Learning in this backend
(`models/rl/qlearning-2026-08-13/model.json` +
`backend/services/adapters/qtableAdapter.js`) -- the backend does the
matrix multiplications itself in plain JavaScript, so it needs **no
PyTorch, no ONNX, and no `onnxruntime-node` dependency at all**.

The cell below also verifies the exported JSON reproduces the exact same
output as the live PyTorch model on random inputs, before you download it.

**Give me the downloaded `best_model_dqn/` folder's two files**
(`weights.json`, `metadata.json`) and I'll place them at
`models/rl/dqn-fold<N>-<date>/` in the backend and write the matching JS
adapter -- no other backend changes needed since the scaler parameters are
unchanged.


In [ ]:
BEST_MODEL_DIR = "best_model_dqn"
os.makedirs(BEST_MODEL_DIR, exist_ok=True)

best_idx = int(results_df["Accuracy"].idxmax())
best_fold_number = int(results_df.iloc[best_idx]["Fold"])
best_row = results_df.iloc[best_idx]
best_agent = dqn_agents[best_idx]
best_fold_data = folds[best_idx]

print("=" * 70)
print("BEST FOLD SELECTION")
print("=" * 70)
print(results_df[["Fold", "Accuracy", "ROC_AUC"]].to_string(index=False))
print(f"\nBest fold: {best_fold_number} (accuracy {best_row['Accuracy']:.4f})")

model_path = os.path.join(BEST_MODEL_DIR, "model.pth")
torch.save(best_agent.policy_network.state_dict(), model_path)

metadata = {
    "model_version": f"dqn-fold{best_fold_number}-{datetime.now(timezone.utc).strftime('%Y-%m-%d')}",
    "algorithm": "DQN",
    "feature_order": CANONICAL_FEATURE_ORDER,
    "preprocessing": {
        "note": "Trained on the 15-dimension representation produced by backend/services/preprocessingService.js. Reuses the deployed models/rl/preprocessing_params.json scaler unchanged.",
        "trained_state_dimension": STATE_SIZE,
    },
    "training_metadata": {
        "fold": best_fold_number,
        "num_folds": len(folds),
        "episodes": NUM_EPISODES,
        "weight_decay": WEIGHT_DECAY,
        "dropout": DROPOUT,
        "trained_on": datetime.now(timezone.utc).strftime("%Y-%m-%d"),
    },
    "evaluation_metadata": {
        "accuracy": float(best_row["Accuracy"]),
        "precision": float(best_row["Precision"]),
        "recall": float(best_row["Recall"]),
        "f1": float(best_row["F1"]),
        "roc_auc": None if pd.isna(best_row["ROC_AUC"]) else float(best_row["ROC_AUC"]),
    },
    "runtime_compatibility": {
        "framework": "javascript",
        "export_format": "layers-json",   # distinct from Q-Learning's "json" (q-table) -- this is a layer-list neural net
        "weights_file": "weights.json",
        "input_dim": STATE_SIZE,
        "num_classes": 2,
        "output_activation": "none",   # DQN outputs raw Q-values -- backend applies softmax itself
    },
}


def export_layers_to_json(sequential_module):
    """Walks an nn.Sequential in forward-pass order and emits a plain JSON-
    serializable layer list. Dropout is skipped -- it's a no-op at eval()
    time, so it contributes nothing to inference and needs no JS equivalent."""
    layers = []
    for module in sequential_module:
        if isinstance(module, nn.Linear):
            layers.append({
                "type": "linear",
                # weight is [out_features][in_features] (PyTorch's own layout) --
                # the JS forward pass does out[i] = bias[i] + sum_j weight[i][j] * x[j]
                "weight": module.weight.detach().cpu().numpy().tolist(),
                "bias": module.bias.detach().cpu().numpy().tolist(),
            })
        elif isinstance(module, nn.ReLU):
            layers.append({"type": "relu"})
        elif isinstance(module, nn.Dropout):
            continue   # inference-time no-op, deliberately not exported
        else:
            raise ValueError(f"Unsupported layer type for JSON export: {type(module)}")
    return layers


best_agent.policy_network.eval()
dqn_layers = export_layers_to_json(best_agent.policy_network.model)

weights_path = os.path.join(BEST_MODEL_DIR, "weights.json")
with open(weights_path, "w") as f:
    json.dump({"input_dim": STATE_SIZE, "layers": dqn_layers}, f)

metadata_path = os.path.join(BEST_MODEL_DIR, "metadata.json")
with open(metadata_path, "w") as f:
    json.dump(metadata, f, indent=2)

print(f"\nSaved: {weights_path}")
print(f"Saved: {metadata_path}")


# --- Verify the exported JSON reproduces the live model exactly ---------
def numpy_forward(layers, x):
    for layer in layers:
        if layer["type"] == "linear":
            W = np.array(layer["weight"])   # [out, in]
            b = np.array(layer["bias"])     # [out]
            x = W @ x + b
        elif layer["type"] == "relu":
            x = np.maximum(0, x)
    return x


with torch.no_grad():
    for _ in range(5):
        sample = np.random.randn(STATE_SIZE).astype(np.float32)
        torch_out = best_agent.policy_network(torch.FloatTensor(sample).unsqueeze(0)).cpu().numpy()[0]
        json_out = numpy_forward(dqn_layers, sample.astype(np.float64))
        max_diff = np.abs(torch_out - json_out).max()
        assert max_diff < 1e-4, f"Exported JSON weights diverge from the live model! max diff = {max_diff}"

print("Verified: exported JSON weights reproduce the live PyTorch model's output exactly.")

try:
    from google.colab import files
    files.download(weights_path)
    files.download(metadata_path)
    print("\nDownload triggered for both files.")
except ImportError:
    print(f"\nNot in Colab -- files are available locally under {BEST_MODEL_DIR}/")
